<a href="https://colab.research.google.com/github/vikassingh0593/bytemaster_stocks/blob/dev/data_aggregation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Update package lists silently
!apt-get update -qq > /dev/null

# Install OpenJDK 11 (required for Spark)
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Download Spark 3.1.1 with Hadoop 3.2
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz

# Extract the downloaded Spark archive
!tar xf spark-3.1.1-bin-hadoop3.2.tgz

# Install the 'findspark' library for easy integration
!pip install -q findspark

# Set environment variables for Java and Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"  # Path to OpenJDK 11
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"  # Path to Spark

# Initialize findspark and PySpark
import findspark
findspark.init()

from pyspark.sql import SparkSession
# Create a Spark session
spark = SparkSession.builder.master("local[*]").appName("MySparkApp").getOrCreate()

# Verify the Spark session
print("Spark version:", spark.version)  # Print the Spark version to confirm setup

spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pyspark.sql.functions import col, when, last, monotonically_increasing_id, lag, lead, coalesce, lit
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql import DataFrame

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lower
from pyspark.sql import functions as F

# Install PySpark and yfinance
!pip install pyspark yfinance

import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

import os
from google.colab import drive
!pip install yfinance
# !pip install --upgrade numpy
from yfinance import Ticker
spark.conf.set("spark.sql.debug.maxToStringFields", 1000) # Or a higher value as needed



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Spark version: 3.1.1


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
check_csv_path = '/content/drive/My Drive/Stocks/check.csv'
check_df = (
    spark.read.csv(check_csv_path, header=True, inferSchema=True)
    .withColumn(
    "NameShareHolder",
    F.lower(F.col("Check")))
    .drop("Check")
)
print(check_df.count())
check_df

46


Investor,NameShareHolder
Rakesh_Jhunjhunwala,nishthajhunjhunwa...
Rakesh_Jhunjhunwala,aryamanjhunjhunwa...
Rakesh_Jhunjhunwala,aryavirjhunjhunwa...
Rakesh_Jhunjhunwala,jhunjhunwalarekha...
Rakesh_Jhunjhunwala,rekhajhunjhunwala
Rakesh_Jhunjhunwala,rekharakeshjhunjh...
Rakesh_Jhunjhunwala,rahuljhunjhunwala
Radhakishan_Damani,brightstarinvestm...
Radhakishan_Damani,radhakishansdamani
Radhakishan_Damani,radhakishansdaman...


In [7]:
comb_path = "/content/drive/My Drive/bytemaster_stocks/2024/March/combined_data.csv"

cmb_df = (
    spark
    .read
    .csv(comb_path, header=True, inferSchema=True)
    .withColumn(
    "NameShareHolder",
    F.lower(F.col("CategoryNameoftheShareholders")))
    .withColumn(
    "NameShareHolder",
    F.regexp_replace(F.col("NameShareHolder"), " ", ""))
    .select("Title", col("Number").cast("int").alias("SecurityCode"), "NameShareHolder", col("ShareholdingPerc").cast("double").alias("ShareholdingPerc"), "NoOfFullyPaidShares", "Year", "Qtr"  )
    .drop_duplicates()
)

cmb_df

Title,SecurityCode,NameShareHolder,ShareholdingPerc,NoOfFullyPaidShares,Year,Qtr
Kapil Raj Finance...,539679,amishayadav,6.66,728887,2024,March
Mudra Financial S...,539819,ashokkotwani,3.81,191000,2024,March
Mudra Financial S...,539819,chandramanimishra,1.96,98000,2024,March
Raghav Productivi...,539837,nishidbabulalshah,1.19,272940,2024,March
Bajaj Healthcare Ltd,539872,vlsfinanceltd,2.04,562610,2024,March
RBL Bank Ltd,540065,vanguardtotalinte...,1.01,6119340,2024,March
Eiko Lifesciences...,540204,aritroashishroy,4.53,627987,2024,March
Tejas Networks Ltd,540595,nipponlifeindiatr...,3.65,6226899,2024,March
Bharat Road Netwo...,540700,nbfcsregisteredwi...,0.11,94169,2024,March
Pro Clb Global Ltd,540703,sangeetasarin,1.75,89100,2024,March


In [8]:
clean_df = (
    check_df
    .join(cmb_df, ["NameShareHolder"], "inner")
)
print(clean_df.count())
clean_df.orderBy("Investor", "title")

317


NameShareHolder,Investor,Title,SecurityCode,ShareholdingPerc,NoOfFullyPaidShares,Year,Qtr
ajayupadhyaya,Ajay_Upadhyaya,BANSWARA SYNTEX L...,503722,1.02,350000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,DCX Systems Ltd,543650,1.21,1350000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,DMCC Speciality C...,506405,1.0,250000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,Dollar Industries...,541403,1.06,600000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,ELECON ENGINEERIN...,505700,1.78,2000000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,GENUS POWER INFRA...,530343,1.58,4800000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,MARAL OVERSEAS LTD.,521018,1.87,775000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,Navin Fluorine In...,532504,1.01,500000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,OMAXE LTD.,532880,1.37,2500000,2024,March
ajayupadhyaya,Ajay_Upadhyaya,Precision Camshaf...,539636,2.11,2000000,2024,March
